In [1]:
import duckdb
import pandas as pd

db_path = r"C:\Users\perna\OneDrive\Documents\RussekLab\projects\poker-project1-RL\data\pluribus.duckdb"
con = duckdb.connect(db_path)

# What columns do we have?
print(con.execute("DESCRIBE preflop").df())

        column_name column_type null   key default extra
0           session     VARCHAR  YES  None    None  None
1              hand      BIGINT  YES  None    None  None
2        player_idx      BIGINT  YES  None    None  None
3       player_name     VARCHAR  YES  None    None  None
4            action     VARCHAR  YES  None    None  None
5            amount      DOUBLE  YES  None    None  None
6    starting_stack      BIGINT  YES  None    None  None
7   finishing_stack      DOUBLE  YES  None    None  None
8            profit      DOUBLE  YES  None    None  None
9        hole_cards     VARCHAR  YES  None    None  None
10         position      BIGINT  YES  None    None  None
11            blind      BIGINT  YES  None    None  None


In [2]:
# How many unique sessions, hands, and players?
print(con.execute("""
    SELECT 
        COUNT(DISTINCT session) as sessions,
        COUNT(DISTINCT session || '-' || CAST(hand AS VARCHAR)) as total_hands,
        COUNT(DISTINCT player_name) as unique_players,
        COUNT(*) as total_decisions
    FROM preflop
""").df())

   sessions  total_hands  unique_players  total_decisions
0        92        10000              14            61811


In [3]:
# Who are the players and how often do they appear?
print(con.execute("""
    SELECT 
        player_name,
        COUNT(*) as decisions,
        COUNT(DISTINCT session || '-' || CAST(hand AS VARCHAR)) as hands_played
    FROM preflop
    GROUP BY player_name
    ORDER BY hands_played DESC
""").df())

   player_name  decisions  hands_played
0     Pluribus      10311          9835
1       MrBlue       9461          8928
2     MrOrange       7672          7396
3         Bill       6942          6609
4       MrPink       6246          5949
5        Eddie       5677          5414
6      MrWhite       4622          4403
7     MrBlonde       2607          2508
8         Budd       2585          2469
9          Joe       1587          1521
10     MrBrown       1395          1349
11     Hattori       1401          1330
12        ORen        793           756
13        Gogo        512           475


In [ ]:
# How often do players fold, call, or raise?
print(con.execute("""
    SELECT 
        action,
        COUNT(*) as count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) as pct
    FROM preflop
    GROUP BY action
    ORDER BY count DESC
""").df())

  action  count   pct
0      f  44077  71.3
1    cbr  10846  17.5
2     cc   6777  11.0
3     sm    111   0.2


In [5]:
# Look at starting stacks across sessions to confirm resets
print(con.execute("""
    SELECT 
        session,
        MIN(hand) as first_hand,
        MAX(hand) as last_hand,
        COUNT(DISTINCT hand) as num_hands,
        MIN(starting_stack) as min_stack,
        MAX(starting_stack) as max_stack
    FROM preflop
    GROUP BY session
    ORDER BY session
    LIMIT 15
""").df())

   session  first_hand  last_hand  num_hands  min_stack  max_stack
0      100           0         70         71      10000      10000
1     100b          71        183        113      10000      10000
2      101           0         91         92      10000      10000
3     101b          92         94          3      10000      10000
4      102           0         72         73      10000      10000
5     102b          73        119         47      10000      10000
6      103           0         66         67      10000      10000
7     103b          67        177        111      10000      10000
8      104           0        109        110      10000      10000
9      105           0         34         35      10000      10000
10     106           0        291        292      10000      10000
11     107           0         57         58      10000      10000
12     108           0        245        246      10000      10000
13     109           0        270        271      10000      1

In [ ]:
# Check continuity between session pairs -- confirmed, 100 -> 100b is a continuous session
print(con.execute("""
    SELECT 
        session,
        MIN(hand) as first_hand,
        MAX(hand) as last_hand
    FROM preflop
    WHERE session IN ('100', '100b', '101', '101b', '40', '40b')
    GROUP BY session
    ORDER BY session
""").df())

  session  first_hand  last_hand
0     100           0         70
1    100b          71        183
2     101           0         91
3    101b          92         94
4      40           0        120
5     40b         121        231


In [7]:
# Create a mapping of folder names to true session IDs
# by stripping the 'b' suffix
print(con.execute("""
    SELECT 
        session,
        REGEXP_REPLACE(session, 'b$', '') as true_session,
        COUNT(DISTINCT hand) as num_hands
    FROM preflop
    GROUP BY session
    ORDER BY true_session, session
    LIMIT 20
""").df())

   session true_session  num_hands
0      100          100         71
1     100b          100        113
2      101          101         92
3     101b          101          3
4      102          102         73
5     102b          102         47
6      103          103         67
7     103b          103        111
8      104          104        110
9      105          105         35
10     106          106        292
11     107          107         58
12     108          108        246
13     109          109        271
14     110          110        195
15     111          111         81
16    111b          111        126
17     112          112         74
18    112b          112        135
19     113          113         73


In [8]:
# Add 'true_session' column to the database
con.execute("""
    CREATE OR REPLACE TABLE preflop AS
    SELECT *,
        REGEXP_REPLACE(session, 'b$', '') as true_session
    FROM preflop
""")

# Verify
print(con.execute("""
    SELECT 
        COUNT(DISTINCT true_session) as true_sessions,
        COUNT(DISTINCT session) as raw_folders
    FROM preflop
""").df())

   true_sessions  raw_folders
0             67           92


In [9]:
# Compute cumulative profit within each true_session per player
# This gives us running stack = 10000 + cumulative profit so far
con.execute("""
    CREATE OR REPLACE TABLE preflop AS
    SELECT *,
        10000 + SUM(profit) OVER (
            PARTITION BY true_session, player_name 
            ORDER BY hand
            ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
        ) as stack_at_decision
    FROM preflop
""")

# Check it looks right
print(con.execute("""
    SELECT true_session, hand, player_name, profit, stack_at_decision
    FROM preflop
    WHERE true_session = '30' AND player_name = 'MrWhite'
    ORDER BY hand
    LIMIT 10
""").df())

  true_session  hand player_name  profit  stack_at_decision
0           30     0     MrWhite   -50.0                NaN
1           30     1     MrWhite     0.0             9950.0
2           30     2     MrWhite     0.0             9950.0
3           30     3     MrWhite     0.0             9950.0
4           30     4     MrWhite     0.0             9950.0
5           30     5     MrWhite  -100.0             9950.0
6           30     6     MrWhite   -50.0             9850.0
7           30     7     MrWhite     0.0             9800.0
8           30     8     MrWhite     0.0             9800.0
9           30     9     MrWhite     0.0             9800.0


In [10]:
# Replace NaN stack_at_decision with 10000 (session starting stack)
con.execute("""
    CREATE OR REPLACE TABLE preflop AS
    SELECT * REPLACE (
        COALESCE(stack_at_decision, 10000) as stack_at_decision
    )
    FROM preflop
""")

# Verify no more NaNs
print(con.execute("""
    SELECT COUNT(*) as remaining_nulls
    FROM preflop
    WHERE stack_at_decision IS NULL
""").df())

   remaining_nulls
0                0


In [12]:
#sanity check
print(con.execute("""
    SELECT 
        COUNT(*) as total_rows,
        COUNT(DISTINCT true_session) as sessions,
        COUNT(DISTINCT player_name) as players,
        ROUND(AVG(stack_at_decision), 0) as avg_stack_at_decision,
        MIN(stack_at_decision) as min_stack,
        MAX(stack_at_decision) as max_stack
    FROM preflop
""").df())

   total_rows  sessions  players  avg_stack_at_decision  min_stack  max_stack
0       61811        67       14                 9860.0   -92531.0    97882.0


In [13]:
print(con.execute("""
    SELECT true_session, player_name, hand, profit, stack_at_decision
    FROM preflop
    WHERE stack_at_decision < 0
    ORDER BY stack_at_decision ASC
    LIMIT 10
""").df())

  true_session player_name  hand  profit  stack_at_decision
0          108    MrOrange   243  3600.0           -92531.0
1          108    MrOrange   238     0.0           -92431.0
2          108    MrOrange   239     0.0           -92431.0
3          108    MrOrange   240     0.0           -92431.0
4          108    MrOrange   241     0.0           -92431.0
5          108    MrOrange   242  -100.0           -92431.0
6          108    MrOrange   234     0.0           -92381.0
7          108    MrOrange   235     0.0           -92381.0
8          108    MrOrange   237   -50.0           -92381.0
9          108    MrOrange   231   325.0           -92136.0


In [14]:
print(con.execute("""
    SELECT true_session, hand, player_name, starting_stack, finishing_stack, profit
    FROM preflop
    WHERE true_session = '30' AND player_name = 'MrWhite'
    ORDER BY hand
    LIMIT 10
""").df())

  true_session  hand player_name  starting_stack  finishing_stack  profit
0           30     0     MrWhite           10000           9950.0   -50.0
1           30     1     MrWhite           10000          10000.0     0.0
2           30     2     MrWhite           10000          10000.0     0.0
3           30     3     MrWhite           10000          10000.0     0.0
4           30     4     MrWhite           10000          10000.0     0.0
5           30     5     MrWhite           10000           9900.0  -100.0
6           30     6     MrWhite           10000           9950.0   -50.0
7           30     7     MrWhite           10000          10000.0     0.0
8           30     8     MrWhite           10000          10000.0     0.0
9           30     9     MrWhite           10000          10000.0     0.0


In [15]:
con.close()

In [16]:
import duckdb
db_path = r"C:\Users\perna\OneDrive\Documents\RussekLab\projects\poker-project1-RL\data\pluribus.duckdb"
con = duckdb.connect(db_path)

# Check min/max stack values
print(con.execute("""
    SELECT 
        MIN(stack_at_decision) as min_stack,
        MAX(stack_at_decision) as max_stack,
        AVG(stack_at_decision) as avg_stack
    FROM preflop
""").df())

# Check MrWhite session 30 still looks right
print(con.execute("""
    SELECT true_session, hand, player_name, finishing_stack, stack_at_decision
    FROM preflop
    WHERE true_session = '30' AND player_name = 'MrWhite'
    ORDER BY hand
    LIMIT 10
""").df())

   min_stack  max_stack    avg_stack
0        0.0    30200.0  9994.916584
  true_session  hand player_name  finishing_stack  stack_at_decision
0           30     0     MrWhite           9950.0            10000.0
1           30     1     MrWhite          10000.0             9950.0
2           30     2     MrWhite          10000.0            10000.0
3           30     3     MrWhite          10000.0            10000.0
4           30     4     MrWhite          10000.0            10000.0
5           30     5     MrWhite           9900.0            10000.0
6           30     6     MrWhite           9950.0             9900.0
7           30     7     MrWhite          10000.0             9950.0
8           30     8     MrWhite          10000.0            10000.0
9           30     9     MrWhite          10000.0            10000.0


In [17]:
# Confirm we only have preflop actions
print(con.execute("""
    SELECT action, COUNT(*) as count
    FROM preflop
    GROUP BY action
    ORDER BY count DESC
""").df())

  action  count
0      f  44077
1    cbr  10846
2     cc   6777
3     sm    111


In [18]:
# Check for nulls across all columns
print(con.execute("""
    SELECT 
        COUNT(*) as total,
        COUNT(session) as session,
        COUNT(hand) as hand,
        COUNT(player_name) as player_name,
        COUNT(action) as action,
        COUNT(hole_cards) as hole_cards,
        COUNT(position) as position,
        COUNT(stack_at_decision) as stack_at_decision,
        COUNT(finishing_stack) as finishing_stack,
        COUNT(true_session) as true_session
    FROM preflop
""").df())

   total  session   hand  player_name  action  hole_cards  position  \
0  61811    61811  61811        61811   61811       61811     61811   

   stack_at_decision  finishing_stack  true_session  
0              61811            61811         61811  


In [19]:
# Check hole card format and length
print(con.execute("""
    SELECT 
        hole_cards,
        LENGTH(hole_cards) as length,
        COUNT(*) as count
    FROM preflop
    GROUP BY hole_cards, LENGTH(hole_cards)
    ORDER BY length DESC
    LIMIT 20
""").df())

   hole_cards  length  count
0        TsQs       4     21
1        4sJh       4     25
2        3d6c       4     21
3        3sJd       4     16
4        Kc9c       4     20
5        QsQd       4     25
6        QsJd       4     23
7        KhQc       4     33
8        6h9h       4     23
9        Qh7c       4     20
10       2d7h       4     19
11       7d4h       4     16
12       Qd3h       4     20
13       KsKh       4     35
14       5c7h       4     19
15       9sKc       4     14
16       Td9s       4     23
17       4cJd       4     23
18       9cTd       4     35
19       7s5h       4     26


In [20]:
# Check position distribution
print(con.execute("""
    SELECT 
        position,
        COUNT(*) as count
    FROM preflop
    GROUP BY position
    ORDER BY position
""").df())

   position  count
0         0  10560
1         1   9071
2         2  10413
3         3  10480
4         4  10624
5         5  10663
